# 🧠 Module 1.1 — Agentic Loops

**Domain 1 · Agentic Architecture & Orchestration** (27% of the exam)
**Task 1.1 · Agentic Loops** · ⏱️ ~45 minutes
**Source:** [claudecertificationguide.com/learn/1-agentic-architecture/1-1-agentic-loops](https://claudecertificationguide.com/learn/1-agentic-architecture/1-1-agentic-loops)

Welcome! This is a hands-on, build-it-yourself companion to the module above. By
the time you reach the bottom, you won't just *know* what an agentic loop is —
you'll have written one, run it against the real Claude API, and personally
watched a classic production bug happen and get fixed.

### 🎯 What you'll build

A working multi-tool agent loop that calls the **real Claude API** — a calculator
and a search tool, chained together, driven entirely by `stop_reason`. No mocks,
no simulations: every response in this notebook comes from Claude.

### ✅ What you'll walk away knowing

1. The deterministic execution cycle that powers every Claude-based agent
2. Why `stop_reason` — not prompts, not vibes — is the one true control signal
3. How to correctly append tool results to conversation history
4. Four real anti-patterns that quietly break agents in production, and why
5. The difference between a safety net and a strategy (iteration caps)

---

> **💳 Heads up — this notebook makes real, billed API calls.** A full run
> through Tasks 2–5 is a handful of short calls (a few cents at most on
> Claude Sonnet 5), but it's not free, and it's not simulated. If you'd rather
> read first and run later, that's a perfectly good way to use this notebook too.

## 🔧 Setup

You'll need the `anthropic` Python package and an API key.

```bash
pip install anthropic
```

Set your key as an environment variable — **never hard-code it in a notebook**,
especially not one you might commit to git or share:

```bash
# Windows (PowerShell)
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# macOS / Linux (bash/zsh)
export ANTHROPIC_API_KEY="sk-ant-..."
```

Then restart the Jupyter kernel so it picks up the new environment variable,
and run the cell below.


In [ ]:
import os
from anthropic import Anthropic

# The SDK reads ANTHROPIC_API_KEY from your environment automatically --
# nothing else to configure here as long as it's set (see the Setup cell above).
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Set it in your shell, then restart the "
        "kernel and run this cell again -- see the Setup section above."
    )

client = Anthropic()
MODEL = "claude-sonnet-5"

print("Connected. Using model:", MODEL)


## 🔑 Key Concept: The Agentic Loop Lifecycle

An **agentic loop** is the core execution cycle behind every Claude-based agent.

> It is **deterministic control flow, defined in code** — not a prompt trick, not
> a retry loop, not a chatbot turn.

### The Four-Step Cycle

| Step | Action |
|---|---|
| 1. Send Request | Submit a request via the Messages API — system prompt, conversation history, prior messages, prior tool results |
| 2. Inspect `stop_reason` | `"tool_use"` → Claude wants to call a tool, loop continues. `"end_turn"` → Claude is done, loop terminates |
| 3. Handle Tool Use | Execute the requested tool(s), **append tool results to conversation history** as a new message, send the updated conversation back to Claude |
| 4. Handle Completion | Present the final response to the user |

### The critical handoff point

> "Step 3 is where loops break. Tool results must be appended to conversation
> history. Miss that, and Claude can't reason about the new information on the
> next iteration."

This one sentence is worth re-reading — a huge fraction of real-world "my agent
got stuck / forgot what it just did" bugs trace back to this exact step.

### `stop_reason` values

The exam focuses on two values, but production systems see more:

| `stop_reason` | Meaning | In exam scope? |
|---|---|---|
| `tool_use` | Claude wants to call one or more tools; loop continues | ✅ |
| `end_turn` | Claude has finished; loop terminates | ✅ |
| `pause_turn` | Continue — long-running server-tool operation in progress | Advanced |
| `max_tokens` | Response hit the token limit | Advanced |
| `stop_sequence` | A custom stop sequence was triggered | Advanced |
| `refusal` | The model declined the request | Advanced |
| `model_context_window_exceeded` | Context limit reached | Advanced |

**Best practice:** treat any non-`end_turn` value as *"not finished — investigate
why"*, rather than assuming only two values will ever show up in production.


## 🧭 Model-Driven vs. Programmatic Decision-Making

**Model-driven approach (exam preference):**
- Claude reads the task and available tools
- The model selects which tool to call based on context
- Adapts to unforeseen situations and edge cases
- Supports flexible tool chaining in sequences you didn't hard-code

**Pre-configured decision trees (limited use):**
- The developer hard-codes which tool runs when
- Rigid but fully deterministic
- Justified only when business logic *demands* compliance (financial ops, security,
  regulatory requirements) — not a general-purpose pattern


## 🛠️ Build Exercise — Task 1: Set Up Tool Definitions

We define two tools and register them with a JSON Schema `input_schema`, exactly
as the real Messages API expects in its `tools` parameter:

- **`calculator`** — accepts a math expression, returns the numeric result
- **`web_search`** — accepts a query, returns a search result

**Why this matters:** this is what exposes *model-driven* decision-making — Claude
looks at the two tool descriptions and the user's request, and decides for itself
which tool (if any) to call, and in what order.

> `web_search` below is a small placeholder lookup rather than a real search API —
> wiring up Brave Search, Tavily, or an internal index is a separate integration
> with its own API key, and orthogonal to what this module teaches. Swap in the
> real thing whenever you're ready; the loop around it doesn't change at all.


In [ ]:
# --- Tool implementations -------------------------------------------------
# Claude never executes code directly -- it only ever *asks* (via a tool_use
# block) for one of these to be run, and your code is what actually calls it.

def calculator(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression and return the result as text."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return f"Error: expression contains disallowed characters: {expression!r}"
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as exc:
        return f"Error evaluating expression: {exc}"


def web_search(query: str) -> str:
    """A placeholder search tool -- see the note above on swapping in a real one."""
    fake_index = {
        "anthropic hq employee count": "Anthropic's San Francisco HQ has approximately 340 employees.",
    }
    key = query.strip().lower()
    return fake_index.get(key, f"No results found for query: {query!r}")


# --- Tool registration, exactly as the Messages API expects ---------------
TOOLS = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression (e.g. '340 * 2') and return the numeric result.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A basic arithmetic expression to evaluate, e.g. '12 * (3 + 4)'.",
                }
            },
            "required": ["expression"],
        },
    },
    {
        "name": "web_search",
        "description": "Search the web for a short factual query and return a text summary of the top result.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query."}
            },
            "required": ["query"],
        },
    },
]

# A dispatch table so our loop can call the right Python function by tool name.
TOOL_FUNCTIONS = {
    "calculator": calculator,
    "web_search": web_search,
}

print("Registered tools:", [t["name"] for t in TOOLS])


## 🛠️ Build Exercise — Task 2: Core Loop Structure

A `while True` loop that calls the **real** `client.messages.create(...)` and
inspects `response.stop_reason`. This skeleton is intentionally *empty* of
tool-handling logic -- we fill that in in Tasks 3 and 4 -- so it makes exactly
**one real API call** and stops.

**Why this matters:** this is the shape the exam expects — the loop's *only*
job at this stage is to send the request and read `stop_reason`. Everything else
branches off of that one field.


In [ ]:
def run_agentic_loop_skeleton(user_prompt: str, system: str) -> None:
    messages = [{"role": "user", "content": user_prompt}]

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system,
            messages=messages,
            tools=TOOLS,
        )

        print("stop_reason =", repr(response.stop_reason))

        # Task 3 and Task 4 logic will go here. For now, just look, then stop --
        # this keeps today's demo to a single real API call.
        break


run_agentic_loop_skeleton(
    "What's double Anthropic's HQ employee count?",
    system="You are a helpful assistant. Briefly narrate your plan before each tool call.",
)


## 🛠️ Build Exercise — Task 3: Handle the `tool_use` Stop Reason

When `stop_reason == "tool_use"`, we must:

1. Extract the `tool_use` block(s) from the response
2. Execute the corresponding Python function
3. Build a `tool_result` message in the correct format
4. **Append both** the assistant's response *and* the tool result message to
   `messages` — this is the "critical handoff point" the module calls out.

Skipping step 4 (or getting the message roles/shape wrong) is the single most
common way agentic loops break in practice.


In [ ]:
def handle_tool_use(response, messages: list) -> None:
    """Execute every tool_use block in `response` and append the right messages.

    Mutates `messages` in place -- this is exactly what you'd do against the
    real SDK in production code.
    """
    # 1) The assistant's own turn (text + tool_use blocks) must be added to
    #    history exactly as Claude returned it, so it can see its own prior
    #    reasoning on the next iteration.
    assistant_content = []
    tool_use_blocks = []
    for block in response.content:
        if block.type == "text":
            assistant_content.append({"type": "text", "text": block.text})
        elif block.type == "tool_use":
            assistant_content.append(
                {"type": "tool_use", "id": block.id, "name": block.name, "input": block.input}
            )
            tool_use_blocks.append(block)

    messages.append({"role": "assistant", "content": assistant_content})

    # 2) Execute each requested tool and build one tool_result block per call.
    tool_result_blocks = []
    for block in tool_use_blocks:
        fn = TOOL_FUNCTIONS[block.name]
        result_text = fn(**block.input)
        print(f"  -> executed tool {block.name}({block.input}) = {result_text!r}")
        tool_result_blocks.append(
            {
                "type": "tool_result",
                "tool_use_id": block.id,   # links the result back to the specific tool_use call
                "content": result_text,
            }
        )

    # 3) Tool results go back as a *user*-role message -- this is what lets
    #    Claude reason about the new information on the next loop iteration.
    messages.append({"role": "user", "content": tool_result_blocks})


## 🛠️ Build Exercise — Task 4: Handle the `end_turn` Stop Reason

When `stop_reason == "end_turn"`, Claude has finished. We extract the final text
and exit the loop — no further API calls needed.


In [ ]:
def extract_final_text(response) -> str:
    """Pull the text out of a final (end_turn) response."""
    return "".join(block.text for block in response.content if block.type == "text")


### 🔗 Putting Tasks 2–4 together: the complete loop

This is the real, non-placeholder version of `run_agentic_loop_skeleton` above,
now against the live API end-to-end. It also folds in **Task 6's safety cap**
(`max_iterations`) up front, since it's one small `if` inside the same loop --
notice that **`stop_reason` alone still drives every branch**; the cap only
exists as a fallback, never as the reason a normal task stops.


In [ ]:
def run_agentic_loop(user_prompt: str, system: str, max_iterations: int = 20):
    """The complete agentic loop: send -> inspect stop_reason -> handle -> repeat.

    Returns (final_text, iterations_used, transcript). `transcript` is every
    raw response object received, in order -- handy for debugging, and reused
    below when we examine the anti-patterns against a real response.
    """
    messages = [{"role": "user", "content": user_prompt}]
    transcript = []
    iterations = 0

    while True:
        iterations += 1

        # Task 6: a safety NET, not the plan -- see the anti-patterns section
        # below for exactly why that distinction matters.
        if iterations > max_iterations:
            print(f"WARNING: hit MAX_ITERATIONS={max_iterations} without an end_turn. Aborting.")
            return None, iterations, transcript

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system,
            messages=messages,
            tools=TOOLS,
        )
        transcript.append(response)

        print(f"[iteration {iterations}] stop_reason = {response.stop_reason!r}")

        if response.stop_reason == "tool_use":
            handle_tool_use(response, messages)
            continue  # loop back to Step 1 with updated history

        if response.stop_reason == "end_turn":
            return extract_final_text(response), iterations, transcript

        # Anything else (pause_turn, max_tokens, refusal, ...): don't silently
        # treat it as success. Investigate rather than assume completion.
        raise RuntimeError(f"Unhandled stop_reason: {response.stop_reason!r}")


# 🔒 Why the cap above is a NET, not a strategy -- illustrative only, not run.
#
# Imagine a tool with a bug that always makes Claude want to call it again --
# e.g. a paginated "fetch_next_page" tool that never correctly reports
# "no more pages left":
#
#     def buggy_fetch_next_page(cursor):
#         return f"Here are 10 more results. cursor={cursor}"   # never says "done"
#
# Without max_iterations, run_agentic_loop() above would call the real, billed
# API forever. There's no benefit to actually proving that against a live
# endpoint -- the `if iterations > max_iterations` check a few lines up is
# exactly what stops this scenario from running away on you in production.


## 🛠️ Build Exercise — Task 5: Test Sequential Tool Calls

**Success criteria:** at least two tool-call iterations before `end_turn`, with
the agent using the first tool's output to inform the second tool call (search →
calculate, not two independent, unrelated calls).

This cell makes real API calls — typically 2–4 of them for this prompt.


In [ ]:
final_answer, n_iterations, transcript = run_agentic_loop(
    "What's double Anthropic's HQ employee count?",
    system="You are a helpful assistant. Briefly narrate your plan before each tool call.",
)

print()
print("Final answer:", final_answer)
print("Total API round-trips:", n_iterations)
print()

if n_iterations >= 3:
    print("That's real sequential tool use: search, then calculate, then end_turn.")
else:
    print("Claude wrapped this up in fewer round-trips than the script assumed --")
    print("real models don't always take the same path a scripted demo would.")
    print("That's fine and expected: stop_reason still drove every decision, end to end.")


## ⚠️ Three Anti-Patterns to Avoid (plus a bonus fourth)

| # | Anti-pattern | Why it fails | Fix |
|---|---|---|---|
| 1 | Parsing natural language ("I'm done", "task complete") | Ambiguous — Claude might say "I've finished the first file" while intending to continue | Rely on `stop_reason` |
| 2 | Arbitrary iteration caps as the *primary* control | Either cuts off useful work or wastes iterations on a task that finished early | Use `stop_reason` as primary control; caps only as a safety net (Task 6) |
| 3 | Content-type checking (`response.content[0].type == "text"`) | Claude can return text *alongside* a `tool_use` block in the same response | Check `stop_reason`, regardless of what content types are present |
| 4 | Forcing `tool_choice: "any"` to avoid text-only turns | Makes `end_turn` structurally impossible — the loop can never terminate itself | Let the model signal completion naturally |

Below, each anti-pattern is written out as **real, working code** — then
**commented out**, so you can see exactly what the mistake looks like without
it ever actually running. The one exception is anti-pattern 3, which we keep
active for one cell so we can catch it red-handed on a real response your own
API call received above.

### 📖 Case Study: The Premature Termination Bug

A customer support agent works fine for simple queries but stops mid-task on
complex ones. The buggy code checks `response.content[0].type == "text"` to
decide whether the agent is finished. When Claude returns explanatory text
*and* a `tool_use` block in the same turn — which is exactly what may have just
happened in your own Task 5 run above — the buggy check sees text at position
`[0]`, concludes the agent is done, and silently drops the tool call.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 1 — Parsing natural language to detect "done"
# ============================================================
# Commented out on purpose: built here so you can see the shape of the
# mistake, not just read a description of it.
#
# def is_done_antipattern_1(response) -> bool:
#     final_text = extract_final_text(response)
#     done_phrases = ("i'm done", "task complete", "finished")
#     return any(phrase in final_text.lower() for phrase in done_phrases)
#
# Why it fails: Claude might say "I've finished analyzing the first file" while
# fully intending to keep going on the next one. Natural language carries no
# fixed contract -- stop_reason does. No phrase list can fix this.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 2 — Iteration cap used as PRIMARY control
# ============================================================
# Commented out on purpose.
#
# def run_agentic_loop_antipattern_2(user_prompt):
#     messages = [{"role": "user", "content": user_prompt}]
#     for i in range(10):                       # <- "stop after 10" IS the plan
#         response = client.messages.create(
#             model=MODEL, max_tokens=1024, messages=messages, tools=TOOLS,
#         )
#         if response.stop_reason != "tool_use":
#             return extract_final_text(response)
#         handle_tool_use(response, messages)
#     return extract_final_text(response)        # silently returns whatever we have at i=9
#
# Why it fails: 10 is a guess, not a signal. A task that legitimately needs 12
# iterations gets truncated mid-work with no warning; a task that finishes in 3
# proves nothing either way. Compare to run_agentic_loop() above, where the cap
# is 20 but is *never the reason a normal task stops* -- stop_reason is.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 3 — Checking content[0].type instead of stop_reason
# ============================================================
# This is the module's case-study bug. Unlike the others, we keep ONE function
# ACTIVE here (not commented out) so we can prove, on a real response your own
# account just received in Task 5, that it gets the wrong answer.

def is_done_antipattern_3(response) -> bool:
    """DO NOT USE -- kept active only to demonstrate the bug below."""
    return response.content[0].type == "text"


def is_done_correctly(response) -> bool:
    """The fix -- authoritative regardless of what content types are present."""
    return response.stop_reason == "end_turn"


first_real_response = transcript[0]
block_types = [b.type for b in first_real_response.content]

print("Real response #1 from your Task 5 run:")
print("  content block types:", block_types)
print("  stop_reason:        ", first_real_response.stop_reason)
print()
print("anti-pattern 3 says done:", is_done_antipattern_3(first_real_response))
print("correct check says done: ", is_done_correctly(first_real_response))

if "text" in block_types and first_real_response.stop_reason == "tool_use":
    print()
    print("There it is: Claude explained itself in text AND called a tool in the")
    print("same turn. The anti-pattern saw the text and declared victory -- wrongly.")
    print("The correct check saw stop_reason='tool_use' and (correctly) kept going.")
else:
    print()
    print("This particular response didn't happen to pair text with a tool call --")
    print("model behavior varies run to run. Re-run Task 5 a couple of times and")
    print("check transcript[0] again; when it does happen, watch the two checks disagree.")


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 4 — Forcing tool_choice="any" to avoid text-only turns
# ============================================================
# Commented out on purpose.
#
# response = client.messages.create(
#     model=MODEL,
#     max_tokens=1024,
#     messages=messages,
#     tools=TOOLS,
#     tool_choice={"type": "any"},      # <- forces a tool call on every single turn
# )
#
# Why it fails: this makes a plain end_turn response structurally impossible --
# Claude is never allowed to just answer in text. The loop can never terminate
# itself and will run until something else (your MAX_ITERATIONS cap) saves you.


## 🎓 Exam Traps (Common Distractors)

| Trap | Why it's wrong |
|---|---|
| Using `response.content[0].type == 'text'` | Text can appear alongside tool calls; presence doesn't indicate completion |
| Setting an arbitrary cap ("stop after 10 loops") as the *primary* mechanism | Caps address runaway loops, not premature exits — use `stop_reason` instead |
| Parsing "I'm done" / "task complete" phrases | Natural language is ambiguous and unreliable for control flow |
| Forcing `tool_choice: 'any'` to prevent text-only returns | Creates infinite loops — let the model signal completion naturally via `stop_reason` |

### Practice Scenario (from the module)

> An agent terminates prematurely when Claude returns text alongside a tool call.
> The loop checks `response.content[0].type == 'text'` for completion. Complex
> queries return incomplete responses. What should you do?
>
> - A. Add an iteration cap of 15
> - B. Check the `stop_reason` field — continue on `tool_use`, terminate on `end_turn`
> - C. Force `tool_choice: 'any'`
> - D. Parse the response for completion phrases
>
> **Answer: B.** (A addresses a different failure mode; C causes infinite loops;
> D is anti-pattern #1.)


## 🏆 Key Takeaways for Exam Prep

1. **`stop_reason` is authoritative** — never use natural language parsing, content-type
   checks, or iteration counts as *primary* loop control.
2. **Tool results must be appended** to conversation history — miss this and Claude
   can't reason about new information on the next iteration.
3. **Model-driven flexibility** — let Claude pick tools based on context; only override
   with hard-coded decision trees when business logic demands it (compliance, safety).
4. **Safety vs. primary control** — iteration caps are acceptable as safety nets, never
   as the primary stopping mechanism.
5. **Anti-patterns are testable** — expect exam questions that present one of the four
   anti-patterns as a plausible-looking "fix". Learn to recognize and reject them.


---

## 🎉 Quick-Fire Recap — See If It Stuck

You built a real agent loop and watched it think out loud, call tools in
sequence, and finish on its own terms. Before you close this notebook, try
answering each of these from memory — no scrolling back up. Cover the answer
with your hand if you're the honor-system type. 😄

**1. In one sentence: what *is* an agentic loop?**
> 💡 The deterministic, code-defined cycle of *send → check `stop_reason` → act
> → repeat*. No prompt tricks, no vibes — just control flow.

**2. Your loop just got `stop_reason == "tool_use"`. What are you supposed to do,
and what's the one step people forget?**
> 💡 Execute the tool, then **append both** the assistant's turn and the tool
> result to `messages`. The forgetting-to-append part is the classic bug — skip
> it, and Claude loses the plot on the very next call.

**3. Why can't you trust `response.content[0].type == "text"` to mean "we're done"?**
> 💡 Because Claude is perfectly happy to say "let me check that for you" *and*
> call a tool in the same breath. Text at position zero proves nothing about
> completion — only `stop_reason == "end_turn"` does.

**4. Your teammate says "let's just cap it at 10 loops and call it a day." What
do you tell them?**
> 💡 That 10 is a guess dressed up as a plan. Keep the cap — as a safety net —
> but let `stop_reason` be the thing that actually ends the loop.

**5. Why is "just check if it said 'I'm done'" a trap, even though it sounds
reasonable?**
> 💡 Natural language doesn't sign contracts. "I've finished the first file"
> can mean *keep going* just as easily as it can mean *stop*. `stop_reason` is
> unambiguous by design; phrases never are.

**6. When (if ever) is it okay to hard-code which tool runs next, instead of
letting Claude decide?**
> 💡 When the business genuinely requires it — regulated finance, security-critical
> actions, compliance workflows. Otherwise, let the model use its judgment; that's
> the whole point of giving it tools instead of writing a flowchart.

**7. Someone suggests forcing `tool_choice: "any"` so the agent "never just
stops talking." Good idea?**
> 💡 No — great way to build an agent that can *never* say it's finished. If
> `end_turn` is structurally banned, only your iteration cap can save you, and
> that's the anti-pattern from Q4 wearing a disguise.

**8. Bragging rights question — what did *your own* API call actually do in
the anti-pattern 3 demo above?**
> 💡 Depends on the run! If Claude paired text with a tool call, you watched
> two lines of code look at the exact same real response and reach opposite
> conclusions — one wrong, one right. That's not a hypothetical anymore; that's
> a bug you just personally reproduced and fixed.

---

### 🚀 Nice work.

That's Task 1.1 in the books — one domain module down, real API calls made,
a real bug reproduced and fixed with your own hands. Onward to
**1.2 — Multi-Agent Orchestration** whenever you're ready.
